In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.linear_model import LinearRegression

# Read the CSV files
df = pd.read_csv("value_rf_imputed.csv")
tc_metadata = pd.read_csv("thermocouples.csv")

# Convert date column
df["MEASUREDATE"] = pd.to_datetime(df["MEASUREDATE"], dayfirst=True)

# Filter criteria: 
# 1. Date after 2020
# 2. Layers: 6177, 5177, 4727
target_layers = [6177, 5177, 4727]
filtered_tc_ids = tc_metadata[tc_metadata['Z'].isin(target_layers)]['no.'].unique()

df_filtered = df[(df["MEASUREDATE"] > "2020-12-31") & (df["Sl. No."].isin(filtered_tc_ids))].copy()

# Store layer mapping for plot titles
tc_layer_map = tc_metadata.set_index('no.')['Z'].to_dict()

# Create folder to store plots
output_folder = "Thermocouple_Plots_Advanced"
os.makedirs(output_folder, exist_ok=True)

# Get unique thermocouples
thermocouples = df_filtered["Sl. No."].unique()
print(f"Total thermocouples found in layers {target_layers}: {len(thermocouples)}")

# Custom start date for extrapolation
start_extrap_date = pd.to_datetime("2014-01-01")

# List to store intersection results
intersection_results = []

for tc in thermocouples:
    # Filter and sort data for the current thermocouple
    tc_data = df_filtered[df_filtered["Sl. No."] == tc].sort_values("MEASUREDATE").dropna(subset=["VALUE"])
    
    if len(tc_data) < 2:
        continue
        
    y = tc_data["VALUE"].values
    x_train = np.arange(len(y)).reshape(-1, 1)
    
    # 1. Advanced Forecast: Exponential Smoothing
    forecast_days_forward = 365 * 10
    try:
        model_es = ExponentialSmoothing(y, trend='add', seasonal='add', seasonal_periods=365)
        fit_es = model_es.fit()
        forecast_es = fit_es.forecast(forecast_days_forward)
    except:
        try:
            model_es = ExponentialSmoothing(y, trend='add')
            fit_es = model_es.fit()
            forecast_es = fit_es.forecast(forecast_days_forward)
        except:
            forecast_es = np.full(forecast_days_forward, y[-1])

    # 2. Linear Fit Extrapolation
    model_lr = LinearRegression()
    model_lr.fit(x_train, y)
    
    m = model_lr.coef_[0]
    c = model_lr.intercept_
    
    # Backward/Forward plot range
    first_data_date = tc_data["MEASUREDATE"].min()
    days_backward = (first_data_date - start_extrap_date).days
    x_plot = np.arange(-days_backward, len(y) + forecast_days_forward).reshape(-1, 1)
    line_lr_full = model_lr.predict(x_plot)
    
    # 3. Intersection Calculation: When Linear Fit hits 115 degrees
    # Linear Equation: y = mx + c. To find x where y = 115:
    if abs(m) > 1e-9:
        # Solving mx + c = 115 => x = (115 - c) / m
        # Note: c here is y at x=0 (the first data point in tc_data)
        days_from_first_point = (115 - c) / m
        intersection_date = first_data_date + pd.Timedelta(days=float(days_from_first_point))
    else:
        intersection_date = None

    layer = tc_layer_map.get(tc, "Unknown")
    intersection_results.append({
        "Sl. No.": tc,
        "Layer": layer,
        "Intersection_Date_115C": intersection_date,
        "Trend_Slope": m
    })

    # -------- Create timelines --------
    dates_all = pd.date_range(start=start_extrap_date, periods=len(x_plot), freq='D')
    dates_hist = tc_data["MEASUREDATE"]
    dates_future = pd.date_range(start=dates_hist.max() + pd.DateOffset(days=1), periods=forecast_days_forward, freq='D')
    
    # Use numpy to concatenate for plotting
    dates_all_vals = np.concatenate([dates_hist.values, dates_future.values])
    
    # -------- Plot --------
    plt.figure(figsize=(12,7))
    plt.plot(dates_hist.values, y, label="Actual Data (Post-2020)", color='blue', alpha=0.7)
    #lt.plot(dates_future.values, forecast_es, linestyle='--', color='red', label="Advanced Forecast (Forward)")
    plt.plot(dates_all.values, line_lr_full, linestyle=':', color='green', label="Linear Fit Extrapolated (2014-2031)", linewidth=2)
    
    # Mark intersection point if it's within plot range
    if intersection_date and start_extrap_date <= intersection_date <= dates_future.max():
        plt.axhline(y=115, color='orange', linestyle='-', alpha=0.3, label="115°C Target")
        plt.plot(intersection_date, 115, 'ro', label=f"115°C hit on {intersection_date.strftime('%Y-%m-%d')}")

    plt.xlabel("Date")
    plt.ylabel("Temperature")
    plt.title(f"TC {tc} (Layer {layer}) | Forecast & 115°C Hit Date")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.xticks(rotation=45)
    plt.xlim(start_extrap_date, dates_future.max())
    plt.tight_layout()
    
    plt.savefig(f"{output_folder}/TC_{tc}.png", dpi=200)
    plt.close()

# Save results to CSV
results_df = pd.DataFrame(intersection_results)
results_df.to_csv("intersection_dates_115C.csv", index=False)
print(f'All plots saved in: {output_folder}')
print(f'Intersection dates saved in: intersection_dates_115C.csv')

# Display top hits (upcoming or historical)
print("\n--- Top 10 Intersection Dates at 115°C ---")
print(results_df.dropna(subset=['Intersection_Date_115C']).sort_values('Intersection_Date_115C').head(10))

Total thermocouples found in layers [6177, 5177, 4727]: 147


c:\Users\sahoo\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\Users\sahoo\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\Users\sahoo\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\Users\sahoo\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\Users\sahoo\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\Users\sahoo\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to co

OutOfBoundsTimedelta: seconds=9964146450207727616, milliseconds=0, microseconds=0, nanoseconds=0